In [1]:
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import torch

# Load data
df = pd.read_csv("../data/clean_articles.csv", encoding='utf-8-sig')
embeddings = np.load("../data/embeddings.npy")

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2', 
    device=device
)

# Connect to ChromaDB
client = chromadb.PersistentClient(path="../data/chromadb")
collection = client.get_collection("urdu_news")

print("✅ Everything loaded!")
print("Articles:", len(df))
print("DB documents:", collection.count())
print("Model ready on:", device)

c:\Users\User\anaconda3\envs\ultra_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 768.47it/s]


✅ Everything loaded!
Articles: 111860
DB documents: 111860
Model ready on: cpu


In [2]:
# Core ULTRA retrieval function
# This is the heart of your system

def ultra_retrieve(query, top_k=15):
    """
    ULTRA retrieval system
    - Short query (<150 chars): uses CLS pooling on headlines
    - Long query (>=150 chars): uses mean pooling on full content
    """
    
    # Step 1: Determine query type (ULTRA static threshold)
    query_length = len(query)
    query_type = "short" if query_length < 150 else "long"
    
    # Step 2: Generate query embedding
    query_embedding = model.encode(query).tolist()
    
    # Step 3: Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    # Step 4: Format results
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            'rank': i + 1,
            'headline': results['metadatas'][0][i]['headline'],
            'category': results['metadatas'][0][i]['category'],
            'date': results['metadatas'][0][i]['date'],
            'distance': results['distances'][0][i]
        })
    
    return query_type, retrieved

print("✅ Retrieval function ready!")
print("Testing with a sample query...")

# Quick test
qtype, results = ultra_retrieve("عالمی بینک پاکستان")
print(f"Query type: {qtype}")
print(f"Results returned: {len(results)}")
print(f"Top result: {results[0]['headline'][:60]}")

✅ Retrieval function ready!
Testing with a sample query...
Query type: short
Results returned: 15
Top result: عالمی بینک نے پاکستان کی غیر ملکی ادائیگیوں کے بارے میں رپور


In [3]:
# Test with different query types
test_queries = [
    "عالمی بینک پاکستان امداد",          # Short Urdu query
    "کرکٹ ورلڈ کپ پاکستان ٹیم",          # Sports query  
    "اسٹاک مارکیٹ کاروبار معیشت",        # Business query
    "سائنس ٹیکنالوجی جدید ایجادات",      # Science query
]

print("=" * 60)
for query in test_queries:
    qtype, results = ultra_retrieve(query, top_k=15)
    print(f"\n🔍 Query: {query}")
    print(f"   Type: {qtype} | Results: {len(results)}")
    print(f"   Top 3 results:")
    for r in results[:3]:
        print(f"   {r['rank']}. [{r['category']}] {r['headline'][:50]}")
    print("-" * 60)


🔍 Query: عالمی بینک پاکستان امداد
   Type: short | Results: 15
   Top 3 results:
   1. [Business & Economics] واشنگٹن پاکستان کی500 ملین ڈالرکی ترقیاتی امداد ور
   2. [Business & Economics] عالمی بینک کا برس بعد پاکستان کی بجٹ امداد بحال کر
   3. [Business & Economics] کورونا وائرسعالمی اداروں کی جانب سے پاکستان کو مال
------------------------------------------------------------

🔍 Query: کرکٹ ورلڈ کپ پاکستان ٹیم
   Type: short | Results: 15
   Top 3 results:
   1. [Sports] انڈر19 ورلڈ کپ ٹیم کا اعلان ایک پاکستانی کھلاڑی بھ
   2. [Sports] کبڈی ورلڈ کپ پاکستان کی 16 رکنی ٹیم کا اعلان تاحال
   3. [Sports] قومی ٹیم تاریخ کے مہنگے ترین کوچنگ اسٹاف کے ہمراہ 
------------------------------------------------------------

🔍 Query: اسٹاک مارکیٹ کاروبار معیشت
   Type: short | Results: 15
   Top 3 results:
   1. [Business & Economics] اسٹاک مارکیٹ میں تیزی ڈالر پھر 152 تک پہنچ گیا سون
   2. [Business & Economics] پاکستان سٹاک ایکسچینج پینتالیس ہزار کی نفسیاتی حد 
   3. [Business & Economics] پی 

In [4]:
# Calculate Precision@15 — your thesis evaluation metric
# Precision@15 = relevant results in top 15 / 15

test_cases = [
    # (query, expected_category)
    ("عالمی بینک پاکستان امداد", "Business & Economics"),
    ("کرکٹ ورلڈ کپ پاکستان ٹیم", "Sports"),
    ("اسٹاک مارکیٹ کاروبار", "Business & Economics"),
    ("فلم اداکار ڈرامہ", "Entertainment"),
    ("انتخابات سیاست حکومت", "Business & Economics"),
    ("فٹبال میچ گول", "Sports"),
    ("موبائل فون ٹیکنالوجی", "Science & Technology"),
    ("کمپیوٹر انٹرنیٹ سافٹ ویئر", "Science & Technology"),
]

print("Precision@15 Results:")
print("=" * 60)

total_precision = 0

for query, expected_cat in test_cases:
    qtype, results = ultra_retrieve(query, top_k=15)
    
    # Count relevant results
    relevant = sum(1 for r in results 
                   if r['category'] == expected_cat)
    precision = relevant / 15
    total_precision += precision
    
    print(f"Query: {query[:35]}")
    print(f"Expected: {expected_cat}")
    print(f"Relevant: {relevant}/15 | P@15: {precision:.2%}")
    print("-" * 60)

avg_precision = total_precision / len(test_cases)
print(f"\n✅ Average Precision@15: {avg_precision:.2%}")
print(f"ULTRA Baseline target:   94.35%")

if avg_precision >= 0.90:
    print("🎉 Excellent! Matches or exceeds baseline!")
else:
    print("📊 Baseline established — extensions will improve this!")

Precision@15 Results:
Query: عالمی بینک پاکستان امداد
Expected: Business & Economics
Relevant: 15/15 | P@15: 100.00%
------------------------------------------------------------
Query: کرکٹ ورلڈ کپ پاکستان ٹیم
Expected: Sports
Relevant: 15/15 | P@15: 100.00%
------------------------------------------------------------
Query: اسٹاک مارکیٹ کاروبار
Expected: Business & Economics
Relevant: 15/15 | P@15: 100.00%
------------------------------------------------------------
Query: فلم اداکار ڈرامہ
Expected: Entertainment
Relevant: 15/15 | P@15: 100.00%
------------------------------------------------------------
Query: انتخابات سیاست حکومت
Expected: Business & Economics
Relevant: 2/15 | P@15: 13.33%
------------------------------------------------------------
Query: فٹبال میچ گول
Expected: Sports
Relevant: 15/15 | P@15: 100.00%
------------------------------------------------------------
Query: موبائل فون ٹیکنالوجی
Expected: Science & Technology
Relevant: 15/15 | P@15: 100.00%
---------------